# Data Cleaning - ShipmentSure

This notebook performs data cleaning on the shipment delivery dataset.

**Steps covered:**
1. Load and inspect the dataset
2. Check for missing values
3. Remove duplicate rows
4. Detect outliers using IQR method
5. Clean column names
6. Save the cleaned dataset

---

In [ ]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## 1. Load and Inspect the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('Train.csv')

# Display basic information
print("="*50)
print("DATASET OVERVIEW")
print("="*50)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn Names: {df.columns.tolist()}")
df.head()

In [ ]:
# Data types and memory usage
print("DATA TYPES:")
print("-"*40)
print(df.dtypes)
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [ ]:
# Descriptive statistics for numeric columns
print("DESCRIPTIVE STATISTICS:")
df.describe()

## 2. Check for Missing Values

In [ ]:
# Check missing values in each column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print("MISSING VALUES SUMMARY:")
print("="*40)
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

if df.isnull().sum().sum() == 0:
    print("\n✅ Great! No missing values found in the dataset.")
else:
    print("\n⚠️ Missing values detected. Handling them below...")

## 3. Remove Duplicate Rows

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    print(f"Removing {duplicates} duplicate rows...")
    df = df.drop_duplicates()
    print(f"New shape after removing duplicates: {df.shape}")
else:
    print("✅ No duplicate rows found.")

## 4. Detect Outliers Using IQR Method

In [ ]:
# Select numeric columns (excluding ID and target)
numeric_cols = ['Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product',
                'Prior_purchases', 'Discount_offered', 'Weight_in_gms']

print("OUTLIER DETECTION (IQR Method):")
print("="*50)

outlier_summary = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({'Column': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR, 'Outliers': outliers})

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

In [ ]:
# Visualize outliers using box plots
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Box Plots for Outlier Detection', fontsize=14, fontweight='bold')

for idx, col in enumerate(numeric_cols):
    row = idx // 3
    col_idx = idx % 3
    sns.boxplot(data=df, y=col, ax=axes[row][col_idx], color='skyblue')
    axes[row][col_idx].set_title(col, fontsize=10)

plt.tight_layout()
plt.show()

## 5. Clean Column Names & Drop ID Column

In [ ]:
# Drop the ID column (not useful for analysis)
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)
    print("✅ Dropped 'ID' column.")

# Rename the target column for clarity
df = df.rename(columns={'Reached.on.Time_Y.N': 'On_Time_Delivery'})
print("✅ Renamed 'Reached.on.Time_Y.N' to 'On_Time_Delivery'.")

print(f"\nFinal columns: {df.columns.tolist()}")
print(f"Final shape: {df.shape}")
df.head()

## 6. Save Cleaned Dataset

In [ ]:
# Create data folder if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save the cleaned dataset
df.to_csv('data/cleaned_data.csv', index=False)
print("✅ Cleaned dataset saved to 'data/cleaned_data.csv'")
print(f"   Rows: {df.shape[0]}, Columns: {df.shape[1]}")

## Summary

| Step | Result |
|------|--------|
| Missing values | None found |
| Duplicates | Checked and removed |
| Outliers | Detected using IQR, visualized with box plots |
| Column cleanup | Dropped ID, renamed target column |
| Output | Saved to `data/cleaned_data.csv` |